In [ ]:
# NewsNinja -> AI News Digest Agent
# This application uses an intelligent multi step approach by LangGraph, where we will be defining simple nodes
# 1. Parser -> Optimize the user's raw topic
# 2. Searcher -> Uses Tavily web search to find the current news articles
# 3. Writer -> Uses Groq LLM to analyze the articles and write the digest

In [ ]:
# q -> Quite Mode -> Hides extra logs
# U -> Upgrade -> Install the latest versions

# tavily-python -> used to search internet in real-time

In [ ]:
!pip install -qU langchain-groq langgraph langchain-community tavily-python

In [ ]:
# Setup APIKeys ->

In [ ]:
# GROQ API Key -> https://console.groq.com/keys
# Tavily -> https://app.tavily.com/home

In [ ]:
import os
import getpass

print("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = getpass.getpass()

print("Enter your Tavily API Key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass()

Enter your Groq API key: 
··········
Enter your Tavily API Key: 
··········


In [ ]:
# 1. Define Tools ->
# LangChain Tools are functions the agent can use to interact with the outside world

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
@tool
def search_news(query: str) -> str:
  """ Search the web for the latest news articles on a given topic"""
  search = TavilySearchResults(
      max_results = 5,   # only returns 5 articles
      search_depth = "advanced", # Do deeper search
      include_answer = True   # Also include summarized answer
  )
  results = search.invoke({"query" : query})

  if not results:
    return "No Result found for the given query"

  formatted = []
  i = 1
  for result in results:
    title = result.get("title" , "No Title")
    url = result.get("url", "")
    content = result.get("content", "No content available")

    formatted.append(
        f"**Article {i}:** {title}\n"
        f"   URL: {url} \n"
        f"   Summary: {content}\n"
    )
    i += 1
  return "\n".join(formatted)

In [ ]:
# 2. Define Agent Graph

In [ ]:
#StateGraph -> Used to create the graph
# END -> tells where the graph ends

In [ ]:
from datetime import datetime
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# Define State
class AgentState(TypedDict):
  topic : str       # Raw user Topic
  search_query : str   # Optimized search input
  search_results : str # Raw Search Output
  digest : str         # Polished output

# Node 1 : Parser
def parse_topic(state: AgentState) -> dict:
  llm = ChatGroq(model = "llama-3.3-70b-versatile", temperature=0)
  prompt = (
      "You are a search query optimizer. Given the following topic, create a concise,"
      "effective search query to find the latest news about it."
      "Return only the optimized search query, nothing else.\n\n"
      f"Topic: {state['topic']}"
  )
  response = llm.invoke(prompt)
  search_query = response.content.strip()
  print(f"\n[Node:Parse Topic] Optimized Search Query : {search_query}")
  return {"search_query" : search_query}

# Node 2 : Searcher
def search_web(state: AgentState) -> dict:
  query = state.get("search_query", state["topic"])
  print(f"\n[Node: search_web] Searching the Web for: {query}")
  results = search_news.invoke({"query" : query})
  print(f"Found {results.count("Article")} articles")
  return {"search_results" : results}

# Node 3 : Writer
def write_digest(state: AgentState) -> dict:
  llm = ChatGroq(model = "llama-3.3-70b-versatile", temperature=0.7)
  prompt = (
      "You are NewsNinja, an elite AI news analyst. Based on the Following Search"
      "results, create a comprehensive yet concise news digest.\n\n"
      "FORMAT YOUR RESPONSE AS: \n"
      "**NewsNinja Digest**\n"
      f"**Topic** {state['topic']}\n"
      f"**Date** {datetime.now().strftime('%B %d %Y')}\n\n"
      "---\n\n"
      "Then Provide:\n"
      "1. A brife **Executive Summary** (2-3 Sentances)\n"
      "2. Key Headlines** with bullet points\n"
      "3. Analysis & Insights** (What this means)\n"
      "4. **Sources** (list the URLs)\n\n"
      "---\n\n"
      f"SEARCH RESULT: \n{state['search_results']}"
  )
  response = llm.invoke(prompt)
  print("\n[Node: write_digest] Digest Written!")
  return {"digest" : response.content}

# Compile Graph
def build_graph() -> StateGraph:
  graph = StateGraph(AgentState)
  graph.add_node("parser", parse_topic)
  graph.add_node("searcher", search_web)
  graph.add_node("writer", write_digest)

  graph.set_entry_point("parser")
  graph.add_edge("parser", "searcher")
  graph.add_edge("searcher", "writer")
  graph.add_edge("writer", END)

  return graph.compile()

# Build it
app = build_graph()

In [ ]:
# 3. Run NEWSNINJA

In [ ]:
topic = input("Enter a news topic to research: ")

if topic.strip():
  print("=" * 60)
  print(f"Starting the Pipeline for: {topic}")
  print("-" * 60)

  result = app.invoke({"topic" : topic})

  print("\n" + "=" * 60)
  print(result["digest"])
  print("=" * 60)

else:
  print("No topic provided! Please Try Again!!!")

Enter a news topic to research: AI
Starting the Pipeline for: AI
------------------------------------------------------------

[Node:Parse Topic] Optimized Search Query : "AI news" OR "Artificial Intelligence updates" site:bbc.com OR site:nytimes.com OR site:techcrunch.com

[Node: search_web] Searching the Web for: "AI news" OR "Artificial Intelligence updates" site:bbc.com OR site:nytimes.com OR site:techcrunch.com
Found 5 articles

[Node: write_digest] Digest Written!

**NewsNinja Digest**
**Topic** AI
**Date** April 18, 2026

**Executive Summary**: The AI landscape is rapidly evolving with significant advancements in generative AI, machine learning, and natural language processing. Tech giants such as Meta, Google, and Amazon are investing heavily in AI research and development, while startups are emerging with innovative AI-powered solutions. The use of AI is expanding across various industries, including healthcare, finance, and education, with potential applications in areas like

In [ ]:
# Topic -> parser node -> output(Optimize topic) -> searcher node -> tavily using tool -> results(articles)
# writer -> polish output

In [ ]:
# MCP Servers

In [ ]:
# https://modelcontextprotocol.io/docs/getting-started/intro